# Fake/Suspicious Vietnamese Post Detection
This notebook trains the multimodal deep learning models (Baseline, Advanced, Proposed) for detecting fake posts.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone code from Github or pull latest changes if already cloned
import os
if not os.path.exists('/content/PostDetector'):
    !git clone https://github.com/PNCanh/PostDetector.git /content/PostDetector

%cd /content/PostDetector
!git pull
!pip install -r requirements.txt


In [ ]:
from config import Config
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    Config.DRIVE_DIR = '/content/drive/MyDrive/PostDetector'
else:
    Config.DRIVE_DIR = Config.BASE_DIR

Config.DATASET_DIR = os.path.join(Config.DRIVE_DIR, 'dataset')
Config.POSTS_DIR = os.path.join(Config.DATASET_DIR, 'posts')

if IN_COLAB:
    import kagglehub
    Config.IMAGES_DIR = kagglehub.dataset_download('cashbowman/ai-generated-images-vs-real-images')
    print('Path to dataset files:', Config.IMAGES_DIR)
else:
    Config.IMAGES_DIR = os.path.join(Config.DATASET_DIR, 'images')

Config.RESOURCES_DIR = os.path.join(Config.DRIVE_DIR, 'resources')
Config.LABELS_FILE = os.path.join(Config.RESOURCES_DIR, 'labels.json')
Config.ABBREVIATION_FILE = os.path.join(Config.RESOURCES_DIR, 'abbreviations.json')
Config.EXPLANATION_LABELS_FILE = os.path.join(Config.RESOURCES_DIR, 'explanation_labels.json')
Config.KEYWORDS_FILE = os.path.join(Config.RESOURCES_DIR, 'keywords.json')
Config.STOPWORDS_FILE = os.path.join(Config.RESOURCES_DIR, 'stopwords.json')
Config.TEENCODE_FILE = os.path.join(Config.RESOURCES_DIR, 'teencode.json')


In [ ]:
# Generate Mutated Data (Optional)
# Run this cell if you want to generate synthetic fake posts before training.
from data.mutation_generator import MutationGenerator

generator = MutationGenerator(Config)
generator.generate_mutated_dataset(num_samples=50)


In [ ]:
# Extract OCR from images
from data.ocr_module import OCRModule

print("Starting OCR extraction...")
ocr = OCRModule(Config)
ocr.process_all_posts()
print("OCR extraction complete.")


In [ ]:
# Prepare Dataset
post_dirs = [os.path.join(Config.POSTS_DIR, d) for d in os.listdir(Config.POSTS_DIR) if os.path.isdir(os.path.join(Config.POSTS_DIR, d))]
print(f"Found {len(post_dirs)} posts.")

# For Baseline/Proposed, we use PhoBERT tokenizer (or XLMR for advanced)
tokenizer = AutoTokenizer.from_pretrained(Config.MODELS['phobert'])
dataset = PostDataset(post_dirs, Config, tokenizer, is_train=True)

In [ ]:
# 1. Train Baseline Model
trainer_baseline = Trainer(BaselineModel, Config, dataset, model_name="Baseline")
baseline_metrics = trainer_baseline.run_cv()

In [ ]:
# 2. Train Advanced Model
tokenizer_adv = AutoTokenizer.from_pretrained(Config.MODELS['xlm_r'])
dataset_adv = PostDataset(post_dirs, Config, tokenizer_adv, is_train=True)
trainer_adv = Trainer(AdvancedModel, Config, dataset_adv, model_name="Advanced")
adv_metrics = trainer_adv.run_cv()

In [ ]:
# 3. Train Proposed Model
# Proposed model uses an ensemble of tokenizers conceptually, but for input ids 
# we can pass the standard one and encode within the model if needed, 
# or simplify by using one shared tokenizer for the fusion.
trainer_prop = Trainer(ProposedModel, Config, dataset, model_name="Proposed")
prop_metrics = trainer_prop.run_cv()

In [ ]:
# Compare Results
all_metrics = {
    'Baseline': baseline_metrics,
    'Advanced': adv_metrics,
    'Proposed': prop_metrics
}

df = save_metrics_table(all_metrics, Config.RESULTS_DIR)
plot_model_comparison(df, Config.RESULTS_DIR)
print("Training Complete! Check /content/output/results for plots and tables.")
display(df)